# Tokenizador

Adaptado de DigitSumTokenizer (run11) para vocabulario de fechas.

In [ ]:
import pandas as pd
import torch
import pickle

class Fechas2Tokenizer():
    """Tokenizador orientado a palabras para fechas2 (basado en DigitSumTokenizer)."""
    
    def __init__(self, train_file):
        # Leer archivo de entrenamiento
        df = pd.read_csv(train_file)
        
        # Extraer vocabulario de palabras
        words = set()
        for text in df['txt']:
            words.update(text.lower().split())
        
        # Crear diccionario palabra->índice
        self.word2index = {
            '<pad>': 0,
            '<sos>': 1,
            '<eos>': 2,
        }
        
        # Añadir palabras ordenadas alfabéticamente
        for i, word in enumerate(sorted(words), start=3):
            self.word2index[word] = i
        
        self.index2word = {v: k for k, v in self.word2index.items()}
        print(f'Vocabulario: {len(self.word2index)} tokens')
    
    def encode(self, x, seq_len=-1):
        """Codifica texto a secuencia de índices (como en run11)."""
        # Añadir tokens especiales
        tokens = [self.word2index['<sos>']]
        
        # Tokenizar palabras
        for word in x.lower().split():
            if word in self.word2index:
                tokens.append(self.word2index[word])
        
        tokens.append(self.word2index['<eos>'])
        
        # Padding si es necesario
        if seq_len > len(tokens):
            tokens = tokens + [self.word2index['<pad>']] * (seq_len - len(tokens))
        
        return torch.tensor(tokens)
    
    def decode(self, x):
        """Decodifica secuencia de índices a texto (como en run11)."""
        if isinstance(x, torch.Tensor):
            x = x.tolist()
        
        words = [self.index2word[i] for i in x if i in self.index2word]
        text = ' '.join(words)
        
        # Limpiar tokens especiales
        text = text.replace('<sos>', '').replace('<eos>', '').replace('<pad>', '')
        return text.strip()

# Crear tokenizador para español
tokenizer_es = Fechas2Tokenizer('../fechas2/fechas2_train.es.csv')

# Guardar
with open('fechas2_tokenizer_es.pkl', 'wb') as f:
    pickle.dump(tokenizer_es, f)
    
print('Tokenizador guardado en fechas2_tokenizer_es.pkl')

# Pruebas del tokenizador

In [ ]:
# Probar el tokenizador
ejemplos = [
    "el siguiente viernes",
    "mañana",
    "pasado mañana",
]

print('Pruebas del tokenizador:\n')
for text in ejemplos:
    encoded = tokenizer_es.encode(text)
    decoded = tokenizer_es.decode(encoded)
    print(f'Original:    {text}')
    print(f'Codificado:  {encoded.tolist()}')
    print(f'Decodificado: {decoded}')
    print()

# Tokenizador para inglés

In [ ]:
# Crear tokenizador para inglés
tokenizer_en = Fechas2Tokenizer('../fechas2/fechas2_train.en.csv')

# Guardar
with open('fechas2_tokenizer_en.pkl', 'wb') as f:
    pickle.dump(tokenizer_en, f)
    
print('Tokenizador guardado en fechas2_tokenizer_en.pkl')

# Tokenizador bilingüe

In [ ]:
class Fechas2BilingualTokenizer():
    """Tokenizador bilingüe que combina español e inglés."""
    
    def __init__(self, train_es_file, train_en_file):
        # Leer ambos archivos
        df_es = pd.read_csv(train_es_file)
        df_en = pd.read_csv(train_en_file)
        
        # Extraer vocabulario combinado
        words = set()
        for text in list(df_es['txt']) + list(df_en['txt']):
            words.update(text.lower().split())
        
        # Crear diccionario
        self.word2index = {
            '<pad>': 0,
            '<sos>': 1,
            '<eos>': 2,
        }
        
        for i, word in enumerate(sorted(words), start=3):
            self.word2index[word] = i
        
        self.index2word = {v: k for k, v in self.word2index.items()}
        print(f'Vocabulario bilingüe: {len(self.word2index)} tokens')
    
    def encode(self, x, seq_len=-1):
        tokens = [self.word2index['<sos>']]
        for word in x.lower().split():
            if word in self.word2index:
                tokens.append(self.word2index[word])
        tokens.append(self.word2index['<eos>'])
        
        if seq_len > len(tokens):
            tokens = tokens + [self.word2index['<pad>']] * (seq_len - len(tokens))
        
        return torch.tensor(tokens)
    
    def decode(self, x):
        if isinstance(x, torch.Tensor):
            x = x.tolist()
        words = [self.index2word[i] for i in x if i in self.index2word]
        text = ' '.join(words)
        text = text.replace('<sos>', '').replace('<eos>', '').replace('<pad>', '')
        return text.strip()

# Crear tokenizador bilingüe
tokenizer_bilingual = Fechas2BilingualTokenizer(
    '../fechas2/fechas2_train.es.csv',
    '../fechas2/fechas2_train.en.csv'
)

# Guardar
with open('fechas2_tokenizer_bilingual.pkl', 'wb') as f:
    pickle.dump(tokenizer_bilingual, f)
    
print('Tokenizador guardado en fechas2_tokenizer_bilingual.pkl')